# 02 — Building an Indexing Pipeline That Retrieves Well

Builds the Azure AI Search index for the five driver's manuals. See `pipeline.md` for the diagram. Run `setup/00_setup.ipynb` first.


## Setup — run once per kernel session


In [ ]:
from helpers import INDEX_NAME, config

print("Search endpoint:", config.search_endpoint)
print("Index name:     ", INDEX_NAME)


---
## Clip 1 (3 min) — Data Source, Skillset, Indexer, Index

Builds the pipeline: OCR the image-only pages, split the text into ~2000-character chunks, and index each chunk as its own search document.


In [ ]:
from helpers import create_data_source, create_index

create_data_source()
create_index()


In [ ]:
from azure.search.documents.indexes.models import SearchIndexerSkillset, SplitSkill
from helpers import (
    SKILLSET_NAME, ai_services_account, build_index_projection, indexer_client,
    inputs, merge_skill, ocr_skill, outputs,
)

split_skill = SplitSkill(
    context="/document",
    text_split_mode=____,
    maximum_page_length=____,
    page_overlap_length=____,
    inputs=inputs(text="/document/merged_text"),
    outputs=outputs(textItems="pages"),
)

skillset = SearchIndexerSkillset(
    name=SKILLSET_NAME,
    skills=[ocr_skill(), merge_skill(), split_skill],
    index_projection=____,
    cognitive_services_account=ai_services_account(),
)
indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{SKILLSET_NAME}' ready.")


In [ ]:
from azure.search.documents.indexes.models import SearchIndexer
from helpers import (
    DATA_SOURCE_NAME, INDEXER_NAME, INDEX_NAME, SKILLSET_NAME, blob_indexing_parameters,
    indexer_client, run_and_wait, title_field_mapping,
)

indexer = SearchIndexer(
    name=INDEXER_NAME,
    data_source_name=____,
    target_index_name=____,
    skillset_name=____,
    parameters=blob_indexing_parameters(),
    field_mappings=[title_field_mapping()],
)
indexer_client.create_or_update_indexer(indexer)
result = run_and_wait(reset=True)


In [ ]:
from helpers import search_client, show_hits

results = search_client.search(search_text="FLAMMABLE radioactive placard", top=3, select=["title", "chunk"])
show_hits(results, fields=("title", "chunk"))


---
## Clip 2 (4 min) — Integrated Vectorization for Text and Image Content

Adds an embedding skill to the skillset and a vectorizer to the index, so no client-side embedding call is needed.


In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIEmbeddingSkill,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
)
from helpers import add_skill, add_vector_search, config, inputs, outputs, run_and_wait

embedding_skill = AzureOpenAIEmbeddingSkill(
    context="/document/pages/*",
    resource_url=config.openai_endpoint,
    deployment_name=____,
    model_name=____,
    dimensions=____,
    inputs=inputs(text="/document/pages/*"),
    outputs=outputs(embedding="vector"),
)

vectorizer = AzureOpenAIVectorizer(
    vectorizer_name="openai-vectorizer",
    parameters=AzureOpenAIVectorizerParameters(
        resource_url=config.openai_endpoint,
        deployment_name=config.openai_embedding_deployment,
        model_name=config.openai_embedding_model,
    ),
)

add_vector_search(vectorizer)
add_skill(embedding_skill, include_vector=True)
result = run_and_wait(reset=True)


In [ ]:
from azure.search.documents.models import VectorizableTextQuery
from helpers import search_client, show_hits

query = "How much space should I leave in front of me?"  # the manuals say "following distance"

results = search_client.search(
    search_text=None,
    vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=5, fields="vector")],
    top=3,
    select=["title", "chunk"],
)
show_hits(results, fields=("title", "chunk"))


---
## Clip 3 (3 min) — Filters and Semantic Ranking

Adds a filterable `state` field taken from the blob folder name, then a semantic ranker configuration.


In [ ]:
from azure.search.documents.indexes.models import (
    FieldMapping,
    FieldMappingFunction,
    SearchField,
    SearchFieldDataType,
)
from helpers import add_field_mapping, add_index_field, run_and_wait, set_projection

state_field = SearchField(
    name=____,
    type=SearchFieldDataType.String,
    filterable=True,
    facetable=True,
)

# The state is one segment of the blob URL, split on "/".
state_mapping = FieldMapping(
    source_field_name="metadata_storage_path",
    target_field_name=____,
    mapping_function=FieldMappingFunction(
        name="extractTokenAtPosition",
        parameters={"delimiter": "/", "position": ____},
    ),
)

add_index_field(state_field)
add_field_mapping(state_mapping)
set_projection(include_vector=True, include_state=True)
result = run_and_wait(reset=True)


In [ ]:
from helpers import search_client, show_hits

query = "How close to a fire hydrant can I park?"  # California, Virginia, Florida: 15 ft; New Jersey: 10 ft

print("-- unfiltered: states mixed together --")
show_hits(search_client.search(search_text=query, top=5, select=["state", "title", "chunk"]))

print("-- filter='state eq \'new-jersey\'' --")
show_hits(search_client.search(search_text=query, filter="state eq 'new-jersey'", top=5, select=["state", "title", "chunk"]))


In [ ]:
from helpers import search_client, show_hits

control_query = "How many feet before a turn must I signal?"  # 100 feet in all five manuals: no filter needed

print("-- unfiltered control question: the answer does not vary by state --")
show_hits(search_client.search(search_text=control_query, top=5, select=["state", "title", "chunk"]))


In [ ]:
from azure.search.documents.indexes.models import (
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
)
from helpers import add_semantic_search

semantic_config = SemanticConfiguration(
    name="semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name=____),
        content_fields=[SemanticField(field_name=____)],
    ),
)
add_semantic_search(semantic_config)


In [ ]:
from helpers import search_client, show_hits

fire_hydrant_query = "How close to a fire hydrant can I park?"

results = search_client.search(
    search_text=fire_hydrant_query,
    query_type="semantic",
    semantic_configuration_name="semantic-config",
    top=5,
    select=["state", "title", "chunk"],
)
show_hits(results, fields=("state", "title", "chunk"))
